# PrepData — Sistema Especialista + Aprendizado Supervisionado

**Disciplina:** Inteligência Artificial e LFA (Trabalho A1)  
**Grupo:** Edmar e Jaiza

## O problema em uma frase
O **PrepData** recebe quatro características numéricas de um dataset (`n_linhas`, `pct_nulos`, `n_colunas_categoricas`, `desbalanceamento_pct`), prevê o nível de preparo necessário (`baixo`, `medio` ou `alto`) com uma **Árvore de Decisão** e combina essa previsão com fatos de contexto informados pelo usuário (`precisa_explicar` e `poucos_dados`) em um **motor de regras SE/ENTÃO** para recomendar o próximo passo técnico com explicação e fonte.

> **Aviso sobre os dados:** todos os 120 registros usados neste projeto são **simulados** em Python (`gerar_dataset.py`) para fins exclusivamente educacionais e não representam datasets reais.

### Como executar este notebook no Google Colab
Basta clicar em **Ambiente de execução → Executar tudo** (*Runtime → Run all*). A primeira etapa clona o repositório público do GitHub e gera `dataset.csv` do zero com semente fixa (`seed = 42`).

## 1. Preparação do ambiente (clone do repositório no Google Colab)

A célula abaixo clona o repositório do projeto caso ele ainda não esteja presente no ambiente atual e entra na pasta `prepdata-sistema-especialista`. No Google Colab, `pandas`, `numpy` e `scikit-learn` já vêm pré-instalados.

In [ ]:
import os

REPO_URL = "https://github.com/JaizaCarvalho/prepdata-sistema-especialista.git"
REPO_DIR = "prepdata-sistema-especialista"

# Só clona se ainda não estivermos dentro do repositório
if not os.path.exists("gerar_dataset.py"):
    if not os.path.exists(REPO_DIR):
        !git clone {REPO_URL}
    %cd {REPO_DIR}

print("Diretório atual:", os.getcwd())
!ls -la

## 2. Geração do dataset simulado (`gerar_dataset.py`)

Este script gera **120 registros simulados** com semente fixa (`seed = 42`), aplica o critério de pontuação documentado + ~8% de ruído em classes vizinhas (casos de fronteira) e salva em `dataset.csv`.

In [ ]:
!python gerar_dataset.py

### Conferência rápida dos arquivos CSV (`dataset.csv` e `regras.csv`)
Abaixo inspecionamos as primeiras linhas de `dataset.csv` e todas as 5 regras da base de conhecimento em `regras.csv`.

In [ ]:
import pandas as pd

df_dataset = pd.read_csv("dataset.csv")
df_regras = pd.read_csv("regras.csv")

print("=== Primeiros 5 registros de 'dataset.csv' ===")
display(df_dataset.head())

print("\n=== Base de conhecimento em 'regras.csv' ===")
display(df_regras)

## 3. Treinamento e avaliação do modelo supervisionado (`modelo.py`)

O script `modelo.py` lê `dataset.csv`, separa **90 registros para treino (75%)** e **30 para teste (25%)** com `stratify=y` e `random_state=42`, treina uma `DecisionTreeClassifier(max_depth=3, random_state=42)` e exibe:
- Acurácia no conjunto de teste;
- Matriz de confusão na ordem `baixo`, `medio`, `alto`;
- Condições aprendidas pela árvore em formato texto (`export_text`).

In [ ]:
!python modelo.py

## 4. Testes isolados do motor de inferência (`motor.py`)

Seguindo a boa prática de testar a infraestrutura de regras separadamente do modelo de ML (*Rules of Machine Learning*), `motor.py` executa **12 cenários locais** com classes fixas para validar:
- Disparo correto de cada regra (`R1` a `R5`);
- Tratamento explícito de informação ausente (`status = 'nao_decidido'`, perguntando apenas o fato realmente necessário);
- Tratamento de entradas fora do domínio ou com tipo inválido (`status = 'sem_regras'`).

In [ ]:
!python motor.py

## 5. Integração completa: 3 casos novos, experimento e 3 testes com `assert` (`main.py`)

O script `main.py` integra os dois módulos na função `consultar(caso, fatos)`:
1. A árvore de decisão prevê a classe (`baixo`, `medio` ou `alto`) a partir dos 4 atributos do dataset.
2. O motor de regras recebe essa classe prevista + os fatos de contexto (`precisa_explicar`, `poucos_dados`) e devolve a recomendação, a regra aplicada, a fonte e a explicação.

Na saída abaixo, observe especialmente:
- **Caso 1 e Caso 3:** previstos corretamente (`alto -> R4` e `baixo -> R1`).
- **Caso 2:** caso de fronteira (2 pontos no critério do especialista, esperado `medio -> R3`), mas previsto pela árvore como `baixo -> R1`. Esse erro é analisado em detalhes em `docs/analise_casos.md`.
- **Experimento:** mantendo os mesmos atributos do Caso 1 e alterando apenas `precisa_explicar` de `1` para `0`, a classe prevista continua **`alto`**, mas a regra muda de **`R4`** para **`R5`**.
- **3 testes `assert`:** positivo (`R4`), negativo (`R1`) e informação ausente (`nao_decidido` perguntando `precisa_explicar`).

In [ ]:
!python main.py

## 6. Verificação (auditoria) da base de conhecimento (`verificar_base.py`)

O script `verificar_base.py` testa todas as **27 combinações** possíveis (`3 classes × 3 valores de precisa_explicar × 3 valores de poucos_dados`, incluindo `None`) para checar se existem:
- Regras mortas / inalcançáveis (que nunca disparam);
- Lacunas de cobertura (`sem_regras`);
- Combinações pendentes que corretamente solicitam informação ao usuário.

In [ ]:
!python verificar_base.py

## 7. Limitações e Declaração de Ética

### Principais limitações
- **Rótulo gerado sobre os mesmos atributos:** o critério de rotulagem usa os 4 atributos do dataset; o ruído de ~8% e a combinação de múltiplos critérios reduzem a trivialidade, mas não substituem dados reais rotulados de forma independente.
- **Amostra pequena e simulada:** 120 registros (30 no teste) não comprovam eficácia em produção.
- **Recorte didático:** 5 regras cobrem apenas um subconjunto de decisões de preparo de dados.
- **Escopo limitado:** o sistema não trata entradas fora do intervalo observado no treino.

### Declaração de Ética
Este projeto tem finalidade exclusivamente educacional. Todos os dados são simulados e não representam datasets reais. As regras seguem fontes técnicas citadas em `docs/fontes.md`, e a recomendação do sistema não substitui a análise de um profissional de dados. A acurácia obtida com poucos registros simulados não comprova eficácia em casos reais. Ferramentas de IA foram usadas como apoio de estudo e desenvolvimento; todo o código e a documentação foram revisados e compreendidos pelos integrantes do grupo.

### Documentação complementar no repositório
- `docs/estudo_inicial.md`: respostas das 5 perguntas do estudo inicial e das 4 perguntas de análise.
- `docs/analise_casos.md`: tabela completa dos 3 casos novos, discussão do erro no Caso 2 e resultado do experimento.
- `docs/fontes.md`: referências técnicas detalhadas de cada regra (`R1` a `R5`).